In [1]:
import pandas as pd
import glob
import os

VERİLERİ BİRLEŞTİR

In [2]:
# 1. Tüm CSV dosyalarının bulunduğu klasör yolu
dosya_yolu = "C:/Users/kerem/Desktop/Akıllı Trafik/istanbul"

# 2. Klasördeki tüm csv dosyalarının listesini al
csv_dosyalar = glob.glob(os.path.join(dosya_yolu, "*.csv"))

In [3]:
# 3. Her CSV dosyasını okuyup listeye ekle
dataframe_listesi = []
for dosya in csv_dosyalar:
    df = pd.read_csv(dosya)

    dataframe_listesi.append(df)

# 4. Tüm DataFrame'leri tek bir DataFrame'de birleştir
birlesik_df = pd.concat(dataframe_listesi, ignore_index=True)

In [4]:
birlesik_df.to_csv("istanbul_trafik_birlesik.csv", index=False)

In [5]:
birlesik_df.shape

(15411588, 8)

In [6]:
birlesik_df.head()

,DATE_TIME,LATITUDE,LONGITUDE,GEOHASH,MINIMUM_SPEED,MAXIMUM_SPEED,AVERAGE_SPEED,NUMBER_OF_VEHICLES
0,2024-10-01 00:00:00,41.042175,28.921509,sxk96p,6,59,33,25
1,2024-10-01 00:00:00,41.097107,28.789673,sxk3z1,9,95,47,26
2,2024-10-01 00:00:00,41.091614,28.361206,sxk1v2,55,125,84,39
3,2024-10-01 00:00:00,40.992737,28.800659,sxk3pq,32,127,62,28
4,2024-10-01 00:00:00,40.981750,28.734741,sxk3ju,7,117,50,139


In [7]:
birlesik_df.tail()

,DATE_TIME,LATITUDE,LONGITUDE,GEOHASH,MINIMUM_SPEED,MAXIMUM_SPEED,AVERAGE_SPEED,NUMBER_OF_VEHICLES
15411583,2024-09-19 07:00:00,41.201477,28.064575,sx7fqs,66,137,92,48
15411584,2024-09-19 07:00:00,41.058655,28.339233,sxk1sf,40,121,86,31
15411585,2024-09-19 07:00:00,41.190491,28.086548,sx7fr4,59,139,87,68
15411586,2024-09-19 07:00:00,41.042175,28.547974,sxk33x,40,117,79,110
15411587,2024-09-19 07:00:00,41.097107,28.515015,sxk3bc,22,158,91,327


In [8]:
birlesik_df.describe()

,LATITUDE,LONGITUDE,MINIMUM_SPEED,MAXIMUM_SPEED,AVERAGE_SPEED,NUMBER_OF_VEHICLES
count,1.541159e+07,1.541159e+07,1.541159e+07,1.541159e+07,1.541159e+07,1.541159e+07
mean,4.106296e+01,2.888705e+01,2.293023e+01,1.018763e+02,5.658984e+01,1.087757e+02
std,1.009604e-01,3.564324e-01,2.435189e+01,3.686585e+01,2.589543e+01,1.410206e+02
min,4.076752e+01,2.796570e+01,0.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00
25%,4.099823e+01,2.869080e+01,3.000000e+00,7.200000e+01,3.300000e+01,2.000000e+01
50%,4.105865e+01,2.893250e+01,1.000000e+01,1.010000e+02,5.500000e+01,5.800000e+01
75%,4.113007e+01,2.915222e+01,4.200000e+01,1.300000e+02,8.000000e+01,1.390000e+02
max,4.134430e+01,2.963562e+01,2.100000e+02,2.510000e+02,2.100000e+02,2.056000e+03


Gereksiz Sütunlar

In [9]:
birlesik_df = birlesik_df.drop(columns=['LATITUDE', 'LONGITUDE'])

GEOHASH Kısaltma

In [10]:
birlesik_df['GEOHASH_SHORT'] = birlesik_df['GEOHASH'].str[:5]

for col in ['MINIMUM_SPEED', 'MAXIMUM_SPEED', 'AVERAGE_SPEED']:
    birlesik_df[col] = pd.to_numeric(birlesik_df[col], errors='coerce')

birlesik_df = birlesik_df.groupby(['GEOHASH_SHORT', 'DATE_TIME']).agg({
    'MINIMUM_SPEED': 'min',
    'MAXIMUM_SPEED': 'max',
    'AVERAGE_SPEED': 'mean',
    'NUMBER_OF_VEHICLES': 'sum',
}).reset_index()

In [11]:
df=birlesik_df.copy()

ZAMAN ARALIKLARINI BİRLEŞTİR

In [12]:
# Tarih sütununu dönüştür
df['DATE_TIME'] = pd.to_datetime(df['DATE_TIME'],format="%Y-%m-%d %H:%M:%S",errors="coerce")
df = df.dropna(subset=['DATE_TIME'])

# Tarih ve saat bilgisini ayır
df['DATE'] = df['DATE_TIME'].dt.date
df['HOUR'] = df['DATE_TIME'].dt.hour

In [13]:
# Zaman dilimlerini tanımla
def time_period(h):
    if 0 <= h < 6:
        return 'Night'             # 00:00–06:00
    elif 6 <= h < 10:
        return 'Morning_Peak'      # 06:00–10:00
    elif 10 <= h < 16:
        return 'Daytime'           # 10:00–16:00
    elif 16 <= h < 20:
        return 'Evening_Peak'      # 16:00–20:00
    else:
        return 'Late_Evening'      # 20:00–00:00

df['TIME_PERIOD'] = df['HOUR'].apply(time_period)
# Gereksiz sütunları kaldır (LATITUDE, LONGITUDE, DATE_TIME, HOUR)
df = df.drop(columns=['DATE_TIME', 'HOUR'])

In [14]:
for col in ['MINIMUM_SPEED', 'MAXIMUM_SPEED', 'AVERAGE_SPEED']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# GEOHASH, DATE, TIME_PERIOD bazında grupla
grouped_df = df.groupby(['GEOHASH_SHORT', 'DATE', 'TIME_PERIOD'], as_index=False).agg({
    'MINIMUM_SPEED': 'min',          # En düşük hız
    'MAXIMUM_SPEED': 'max',          # En yüksek hız
    'AVERAGE_SPEED': 'mean',         # Ortalama hız
    'NUMBER_OF_VEHICLES': 'sum'      # Toplam araç sayısı
})

# Sayısal sütunları yuvarla ve tamsayıya çevir
grouped_df['AVERAGE_SPEED'] = grouped_df['AVERAGE_SPEED'].round(0).astype(int)

# Yeni CSV olarak kaydet
grouped_df.to_csv("istanbul_traffic_2024_grouped.csv", index=False)

In [15]:
grouped_df.head()

,GEOHASH_SHORT,DATE,TIME_PERIOD,MINIMUM_SPEED,MAXIMUM_SPEED,AVERAGE_SPEED,NUMBER_OF_VEHICLES
0,sx7ch,2024-01-01,Daytime,1,161,65,722
1,sx7ch,2024-01-01,Evening_Peak,2,137,59,789
2,sx7ch,2024-01-01,Late_Evening,1,149,71,531
3,sx7ch,2024-01-01,Morning_Peak,2,135,69,194
4,sx7ch,2024-01-01,Night,5,133,67,217


In [16]:
grouped_df.shape

(271006, 7)

HAVA DURUMUNU BİRLEŞTİR

In [17]:
hava = pd.read_csv("istanbul_hava_durumu_saatlik_2024.csv")

# Hava durumu tarihini dönüştür
hava['Tarih_Saat'] = pd.to_datetime(hava['Tarih_Saat'])
hava['DATE'] = hava['Tarih_Saat'].dt.date  # sadece günü al

# Günlük ortalama hava durumu oluştur
hava_gunluk = hava.groupby('DATE').agg({
    'Sicaklik': 'mean',
    'Yagis': 'mean',
    'Ruzgar_Hizi': 'mean',
    'Nem': 'mean',
    'Bulutluluk': 'mean'
}).reset_index()

# Trafik verisinde tarih sütunu tarih tipine dönüştür
grouped_df['DATE'] = pd.to_datetime(grouped_df['DATE']).dt.date

# Gün bazında birleştir (inner join)
birlesik = pd.merge(grouped_df, hava_gunluk, on='DATE', how='left')



# Sayısal sütunlar
numeric_cols = ["Sicaklik", "Yagis", "Ruzgar_Hizi", "Nem", "Bulutluluk"]

# Hatalı biçimleri düzeltme fonksiyonu
def clean_number(x):
    if pd.isna(x):  # boşsa dokunma
        return None
    x = str(x)
    # Noktaları temizle (binlik ayırıcı sanılmış)
    x = x.replace('.', '')
    # Virgülü noktaya çevir (Avrupa formatı için)
    x = x.replace(',', '.')
    try:
        val = float(x)
        # Gerçekçi olmayan büyük değerleri ölçekle (örnek: 11429166666666600 → 11.43)
        while val > 1000:
            val = val / 1e6
        return round(val, 2)
    except:
        return None

# Tüm sayısal sütunlara uygula
for col in numeric_cols:
    birlesik[col] = birlesik[col].apply(clean_number)

# Tür dönüşümü
birlesik[numeric_cols] = birlesik[numeric_cols].astype(float)

# Kaydet
birlesik.to_csv("trafik_hava_birlesik.csv", index=False)

Yağış Durumu Sınıflandırma

In [18]:
# Mantıksız derecede yüksek (>=100) yağışları 1000’e bölerek düzelt
birlesik["Yagis"] = birlesik["Yagis"].apply(lambda x: x/1000 if x >= 100 else x)

# Yağış durumunu sınıflandır
def classify_rain(x):
    if x == 0:
        return "Yağışsız"
    elif 0 < x < 0.5:
        return "Hafif Yağmur"
    elif 0.5 <= x < 2:
        return "Yağışlı"
    else:
        return "Sağanak"

birlesik["rainfall_condition"] = birlesik["Yagis"].apply(classify_rain)

# Yeni dosya olarak kaydet
birlesik.to_csv("trafik_hava.csv", index=False, encoding="utf-8-sig")

In [19]:
birlesik = birlesik.drop(columns=['Sicaklik', 'Yagis', "Ruzgar_Hizi", "Nem", "Bulutluluk"], errors='ignore')

In [20]:
birlesik.shape

(271006, 8)